# Starter Template: from a CSV file to your first model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cpilsworth/ml2026/blob/main/Starter_Template.ipynb)

A beginner's walk-through of processing a dataset, one step at a time. It follows the
same order as the README and `Starter_Template.py`:

| Part | What happens | When |
|---|---|---|
| **1** | Load and look at the data | Always |
| **2** | Clean the data | Always |
| **3A** | Classification: predict a category | Your data has an answer (target) column |
| **3B** | Clustering: find groups | There is no answer column (or you want to find groups anyway) |

**How to use this notebook**
1. Change the **Settings** cell below (it's the only cell you need to edit).
2. Run the cells **one at a time, from the top**: click a cell and press **Shift + Enter**.
3. Read the text above each cell first; it explains what the step does and why.
4. Screenshot the outputs and charts for your report.

To start again from scratch: **Runtime → Restart session**, then run from the top.

## Setup: load the libraries

Libraries are toolboxes of ready-made code. Colab already has all of these installed.
(On your own computer: `python -m pip install numpy pandas matplotlib scikit-learn`.)

In [ ]:
import numpy as np                  # maths and arrays
import pandas as pd                 # tables (DataFrames)
import matplotlib.pyplot as plt     # charts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, silhouette_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print("Libraries loaded")

## Settings: change these for your own dataset

| Setting | What it means |
|---|---|
| `DATA_FILE` | A web address of a CSV, or the name of a file you upload |
| `SEPARATOR` | What separates the columns: `","` (most files), `"\t"` (tab), `";"` (semicolon). If everything ends up in **one** column, this is wrong. |
| `TARGET_COLUMN` | The column to predict (the answer), e.g. `"Churn"`. Use `None` if there isn't one. |
| `COLUMNS_TO_DROP` | Columns that won't help: IDs, names, free text |
| `NUMBER_OF_CLUSTERS` | Clustering only. Run once, look at the elbow and silhouette charts, then set it. |
| `UPLOAD_MY_OWN_FILE` | `True` shows an upload button in the next cell |

**The example is the exam practice data** (mobile phone price bands). To try the Week 3
mall data (clustering) instead, use:

```python
DATA_FILE = "https://raw.githubusercontent.com/cpilsworth/ml2026/main/Mall_Customers.csv"
SEPARATOR = ","
TARGET_COLUMN = None
COLUMNS_TO_DROP = ["CustomerID", "Gender"]
NUMBER_OF_CLUSTERS = 5
```

In [ ]:
DATA_FILE = "https://raw.githubusercontent.com/cpilsworth/ml2026/main/Practice_Data.csv"
SEPARATOR = "\t"                  # Practice_Data.csv uses tabs
TARGET_COLUMN = "price_range"      # None = no answer column
COLUMNS_TO_DROP = []
NUMBER_OF_CLUSTERS = 4
UPLOAD_MY_OWN_FILE = False         # True = upload a CSV from your computer

### Optional: upload your own file

Only runs if `UPLOAD_MY_OWN_FILE = True`. A **Choose files** button appears; pick your
CSV and it becomes the `DATA_FILE`. Uploaded files are deleted when the Colab session
ends, so upload again next time.

In [ ]:
if UPLOAD_MY_OWN_FILE:
    try:
        from google.colab import files
        uploaded = files.upload()          # shows the "Choose files" button
        DATA_FILE = list(uploaded)[0]      # use the file you just uploaded
        print("Using", DATA_FILE)
    except ImportError:
        print("Not in Colab: put the CSV next to this notebook and set DATA_FILE to its name.")
else:
    print("Using", DATA_FILE)

---
# Part 1: Load and look at the data

## Step 1: Load the file

`pd.read_csv` reads the file into a table called `df` (a "data frame").

- `UnicodeDecodeError`? Add `encoding="latin1"` inside the brackets.
- An Excel file? Use `pd.read_excel("file.xlsx")`.

In [ ]:
df = pd.read_csv(DATA_FILE, sep=SEPARATOR)
print("Rows and columns:", df.shape)

## Step 2: Look at the data BEFORE changing anything

Screenshot these outputs for the report: they show you checked the data.

**First 5 rows.** Do the columns look right? (A table in Colab ends with the
last line of the cell, so there's no `print` here.)

In [ ]:
df.head()

**Column names, types and non-missing counts.** A number column showing as `object`
means it contains text (e.g. `"£5"`) and needs fixing in Step 5.

In [ ]:
df.info()

**Summary statistics.** Look for strange values: a minimum of 0 where 0 is impossible,
or a maximum far too big (e.g. age 999).

In [ ]:
df.describe()

**Missing values and duplicates.**

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())
print("\nNumber of duplicate rows:", df.duplicated().sum())

---
# Part 2: Clean the data

For the report, write down **what** you did at each step and **why**.

## Step 3: Remove duplicate rows

Exact copies of another row add no information and can bias the model.

In [ ]:
rows_before = len(df)
df = df.drop_duplicates()
print("Removed", rows_before - len(df), "duplicate rows")

## Step 4: Drop columns that don't help

ID numbers and names identify a row but don't explain anything, and in clustering they
would even be treated as a measurement.

In [ ]:
df = df.drop(columns=COLUMNS_TO_DROP)
print("Dropped:", COLUMNS_TO_DROP)
print("Columns left:", list(df.columns))

## Step 5: Fix wrong values and messy text (optional)

Depends on your data. Remove the `#` at the start of a line to use an example.

- A number stored as text, e.g. `"£5.99"` → `5.99`
- Messy categories, e.g. `" Male"`, `"male"`, `"MALE"` → `"male"`
- An impossible zero that really means "unknown". In the practice data, 180 phones have a
  screen width (`sc_w`) of 0. Turning those into missing values lets Step 6 fill them sensibly.

In [ ]:
# df["Price"] = pd.to_numeric(df["Price"].str.replace("£", ""), errors="coerce")
# df["Gender"] = df["Gender"].str.strip().str.lower()
# df["sc_w"] = df["sc_w"].replace(0, np.nan)      # np.nan = missing

print("Optional fixes: edit this cell if your data needs them")

## Step 6: Fill in missing values

- Rows with **no answer** can't be used for training, so they are deleted.
- **Number** columns are filled with the **median** (the middle value). Outliers don't affect it.
- **Text** columns are filled with the **mode** (the most common value).

In [ ]:
if TARGET_COLUMN is not None:
    df = df.dropna(subset=[TARGET_COLUMN])

for column in df.columns:
    if df[column].isnull().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[column]):
            df[column] = df[column].fillna(df[column].median())
            print("Filled gaps in", column, "with the median")
        else:
            df[column] = df[column].fillna(df[column].mode()[0])
            print("Filled gaps in", column, "with the most common value")

print("Missing values left:", df.isnull().sum().sum())

## Step 7: Remove outliers (optional)

Only if `describe()` showed impossible or extreme values. The **IQR rule** keeps
values between Q1 − 1.5 × IQR and Q3 + 1.5 × IQR. Replace `"Income"` with your column
and remove the `#`s.

A boxplot helps you spot outliers: `df.boxplot(column="Income"); plt.show()`

In [ ]:
# Q1 = df["Income"].quantile(0.25)
# Q3 = df["Income"].quantile(0.75)
# IQR = Q3 - Q1
# df = df[(df["Income"] >= Q1 - 1.5 * IQR) & (df["Income"] <= Q3 + 1.5 * IQR)]

print("Clean data: rows and columns =", df.shape)

---
# Part 3A: Classification (predict the `TARGET_COLUMN`)

Run this part if your data has an answer column. See `Week2.py` for more models, such
as AdaBoost, bagging and voting.

**No answer column?** The first cell below stops with a message. Scroll down to
**Part 3B**, click its first code cell, and choose **Runtime → Run cell and below**.

## Step 8: Split the inputs (X) from the answer (y)

Models only understand numbers:
- **The answer:** text labels (e.g. `"Yes"`/`"No"`) become 0/1 with a `LabelEncoder`.
- **The inputs:** **one-hot encoding** makes a 0/1 column for each category (e.g.
  `City` → `City_Leeds`, `City_York`). Number columns stay as they are.

In [ ]:
assert TARGET_COLUMN is not None, ("TARGET_COLUMN is None, so there is nothing to predict. "
    "Go to Part 3B, click its first code cell, then Runtime -> Run cell and below.")

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

if not pd.api.types.is_numeric_dtype(y):
    encoder = LabelEncoder()
    y = encoder.fit_transform(y)
    print("Answer labels turned into numbers:",
          dict(zip(encoder.classes_, range(len(encoder.classes_)))))

X = pd.get_dummies(X, drop_first=True, dtype=int)
print("Number of input columns:", X.shape[1])

## Step 9: Check the class balance

If one answer is much more common (e.g. 90% "No"), say so in the report and trust the
**F1 score** more than accuracy. A model that always says "No" would be 90% accurate
and completely useless.

In [ ]:
pd.Series(y).value_counts()

## Step 10: Train/test split

The models learn from 80% of the rows and are tested on the other 20%, which they have
never seen. That gives a fair score.

- `stratify=y` keeps the same mix of answers in both parts.
- `random_state=42` gives the same split every time.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Training rows:", len(X_train), "| test rows:", len(X_test))

## Step 11: Scale the numbers

Puts every column on the same scale (average 0, spread 1), so a column with big numbers
(e.g. salary) doesn't drown out a small one (e.g. age). KNN and logistic regression need
this; trees don't mind either way.

⚠️ **Fit the scaler on the TRAINING data only**, then reuse it on the test data.
Fitting on the test data is **data leakage**.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn the scaling from TRAIN
X_test_scaled = scaler.transform(X_test)         # apply the same scaling to TEST
print("Data scaled")

## Step 12: Train several models and compare them

Each model follows the same three lines: **create → fit (learn) → predict**.

- **Accuracy** = % of answers correct.
- **F1** balances precision and recall. `average="macro"` gives every class equal
  weight, so it also works with more than two classes.

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)          # learn from the training rows
    y_pred = model.predict(X_test_scaled)       # predict the unseen test rows
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    results.append({"Model": name, "Accuracy": round(accuracy, 3), "F1": round(f1, 3)})

results = pd.DataFrame(results).sort_values("F1", ascending=False)
best_name = results.iloc[0]["Model"]
best_model = models[best_name]
print("Best model:", best_name)
results

A bar chart of the scores, which makes a good screenshot for the report:

In [ ]:
results.plot(x="Model", y=["Accuracy", "F1"], kind="bar", figsize=(8, 4), rot=0)
plt.title("Model comparison on the test data")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Step 13: Look more closely at the best model

- The **classification report** gives precision, recall and F1 for each class.
- The **confusion matrix**: rows are the real answer and columns are the prediction. The
  diagonal holds the correct answers; everything else is a mistake. Which classes get mixed up?

In [ ]:
best_pred = best_model.predict(X_test_scaled)

print("Classification report for", best_name)
print(classification_report(y_test, best_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, best_pred))

## Step 14: Which columns matter most?

The random forest scores how much each column helped. This is useful for business
recommendations, e.g. "RAM drives the price band most".

In [ ]:
importance = pd.Series(models["Random Forest"].feature_importances_, index=X.columns)
top10 = importance.sort_values(ascending=False).head(10)

top10.sort_values().plot(kind="barh", figsize=(7, 4))
plt.title("Most important columns (random forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

top10.round(3)

---
# Part 3B: Clustering (find groups)

Run this part when there is no answer column. You can also run it on data that **has**
one: the answer column is left out, so you can see whether the groups match the answers.
See `Week3.py` for Fuzzy C-Means, DBSCAN and hierarchical clustering.

## Step 8: Prepare the inputs

Every remaining column is an input, apart from the answer column (if there is one).
Text columns become 0/1 columns (one-hot encoding).

In [ ]:
if TARGET_COLUMN is not None:
    X_cluster = df.drop(columns=[TARGET_COLUMN])   # clustering doesn't use the answer
else:
    X_cluster = df.copy()

X_cluster = pd.get_dummies(X_cluster, drop_first=True, dtype=int)
print("Columns used for clustering:", list(X_cluster.columns))

## Step 9: Scale the numbers (always do this for clustering)

Clustering measures **distances** between points. Without scaling, a column with big
numbers (e.g. income in thousands) would decide everything.

In [ ]:
cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)
print("Data scaled")

## Step 10: Choose the number of groups (k)

Try k = 2 to 10 and measure:
- **Inertia:** how spread out each group is. Lower is better, but it always falls, so look
  for the **elbow** where it levels off.
- **Silhouette:** how well separated the groups are, from −1 to 1. Higher is better.

Then go back to the **Settings** cell, set `NUMBER_OF_CLUSTERS`, re-run Settings, and
continue from Step 11. Ask how many groups the business could actually use.

In [ ]:
k_values = range(2, 11)
inertias = []
silhouettes = []
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_cluster_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_cluster_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow plot")
axes[0].set_xlabel("Number of groups (k)")
axes[0].set_ylabel("Inertia")
axes[1].plot(list(k_values), silhouettes, marker="o", color="green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Number of groups (k)")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

pd.DataFrame({"k": list(k_values), "Inertia": inertias, "Silhouette": silhouettes}).round(3)

## Step 11: Run K-Means with the chosen k

How to read the silhouette score: above about 0.5 means clear groups, 0.25–0.5
reasonable groups, and near 0 no real groups. The phone data scores only about 0.06,
which tells you phones don't fall into natural groups by their specs. That is a
finding in itself, not a mistake.

In [ ]:
kmeans = KMeans(n_clusters=NUMBER_OF_CLUSTERS, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_cluster_scaled)

print("k =", NUMBER_OF_CLUSTERS)
print("Silhouette score:", round(silhouette_score(X_cluster_scaled, clusters), 3))
print("How many in each group:")
print(pd.Series(clusters).value_counts().sort_index())

## Step 12: Describe each group

This is the most useful output for a business. The averages are in the **original units**
(not scaled), so you can give each group a name, e.g. *"young, low income, high spending"*.

In [ ]:
X_cluster.assign(Cluster=clusters).groupby("Cluster").mean().round(1)

## Step 13: Draw the groups

**PCA** squashes all the columns into 2, so the groups can be drawn on a flat chart. The
two new axes don't mean anything on their own.

To plot two of your own columns instead, e.g. income against spending:
`plt.scatter(df["Annual Income (k$)"], df["Spending Score (1-100)"], c=clusters, cmap="tab10")`

In [ ]:
points_2d = PCA(n_components=2).fit_transform(X_cluster_scaled)

plt.figure(figsize=(6, 5))
plt.scatter(points_2d[:, 0], points_2d[:, 1], c=clusters, cmap="tab10", s=25)
plt.title(f"K-Means groups (k = {NUMBER_OF_CLUSTERS}), drawn with PCA")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.tight_layout()
plt.show()

---
# What next?

**Report checklist** (README section 6):
- [ ] Business problem: what does the company want to know?
- [ ] Data issues found (missing values, duplicates, outliers, imbalance), how you fixed them, and **why**
- [ ] The techniques you used and why they suit this data
- [ ] Results (accuracy / F1 or silhouette) with screenshots
- [ ] A business recommendation
- [ ] Limitations, and what you'd try with more time

**Ideas to go further**
- Classification: try the ensembles in `Week2.py` (AdaBoost, bagging, voting), or
  `cross_val_score` for a fairer average over several splits.
- Clustering: compare K-Means with DBSCAN or Agglomerative clustering (`Week3.py`).
- Shopping baskets: `Week4.py`. Text: `Week6.py`.
- A number to predict (e.g. price)? Swap the classifiers for `LinearRegression` or
  `RandomForestRegressor`, and score with `r2_score`.
- Not sure which analysis fits your data? See `Analysis_and_Model_Selection_Guide.md`.